In [0]:
%sql
CREATE OR REPLACE TEMP VIEW seismic_prep AS
SELECT
    TRIM(event_id) AS event_id,
    TIMESTAMP_MILLIS(CAST(event_timestamp_raw AS BIGINT)) AS event_timestamp,
    DATE(TIMESTAMP_MILLIS(CAST(event_timestamp_raw AS BIGINT))) AS event_date,
    
    CASE 
        WHEN magnitude IS NULL OR TRIM(CAST(magnitude AS STRING)) IN ('NaN', 'null', '') THEN NULL 
        ELSE CAST(magnitude AS DECIMAL(3,1)) 
    END AS magnitude,
    
    UPPER(TRIM(COALESCE(magnitude_type, 'UNKNOWN'))) AS magnitude_type,
    
    CASE 
        WHEN depth IS NULL OR TRIM(CAST(depth AS STRING)) IN ('NaN', 'null', '') THEN NULL 
        ELSE CAST(depth AS DECIMAL(5,1)) 
    END AS depth_km,
    
    CAST(latitude AS DOUBLE) AS latitude,
    CAST(longitude AS DOUBLE) AS longitude,
    
    TRIM(place) AS place,
    LOWER(TRIM(event_type)) AS event_type,
    source_file,
    CAST(ingestion_timestamp AS TIMESTAMP) AS ingestion_timestamp
FROM cr_seismic_analysis.bronze.seismic_raw;

-- Audit test
SELECT 
    typeof(magnitude) AS magnitude_type, 
    typeof(depth_km) AS depth_type, 
    COUNT(*) AS raw_count 
FROM seismic_prep;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW seismic_filtered AS
SELECT *
FROM seismic_prep
WHERE 
    event_id IS NOT NULL AND TRIM(event_id) != ''
    AND event_timestamp IS NOT NULL
    AND magnitude IS NOT NULL AND magnitude > 0.0
    AND magnitude_type IS NOT NULL AND magnitude_type != 'UNKNOWN'
    AND depth_km IS NOT NULL AND depth_km >= 0.0 -- Allows surface-level events (0.0 km)
    AND latitude BETWEEN 8.0 AND 11.5
    AND longitude BETWEEN -86.0 AND -82.5
    AND (place LIKE '%Costa Rica%' OR place LIKE '%, CR%');

-- Audit test
SELECT COUNT(*) AS filtered_count 
FROM seismic_filtered;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW seismic_enriched AS
SELECT
    f.*,
    CASE 
        WHEN f.magnitude < 3.0 THEN 'Micro (< 3.0)'
        WHEN f.magnitude BETWEEN 3.0 AND 3.99 THEN 'Minor (3.0-3.9)'
        WHEN f.magnitude BETWEEN 4.0 AND 4.99 THEN 'Light (4.0-4.9)'
        WHEN f.magnitude BETWEEN 5.0 AND 5.99 THEN 'Moderate (5.0-5.9)'
        WHEN f.magnitude >= 6.0 THEN 'Strong/Major (>= 6.0)'
        ELSE 'Unknown'
    END AS magnitude_category,
    
    ROUND(f.latitude * 4.0) / 4.0 AS lat_bin,
    ROUND(f.longitude * 4.0) / 4.0 AS lon_bin,
    CURRENT_TIMESTAMP() AS silver_processed_at
FROM seismic_filtered f;

-- Audit test
SELECT magnitude_category, COUNT(*) AS category_count 
FROM seismic_enriched 
GROUP BY magnitude_category;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW seismic_deduplicated AS
WITH ranked AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY event_id 
            ORDER BY event_timestamp DESC
        ) AS rn
    FROM seismic_enriched
)
SELECT * FROM ranked WHERE rn = 1;

-- Audit test
SELECT COUNT(*) AS ready_for_silver_count 
FROM seismic_deduplicated;

In [0]:
%sql
WITH typed_data AS (
    SELECT 
        TRIM(event_id) AS event_id,
        magnitude,
        UPPER(TRIM(magnitude_type)) AS magnitude_type,
        magnitude_category AS magnitude_class,
        TRIM(place) AS place,
        event_timestamp,
        longitude,
        latitude,
        depth_km AS depth,
        source_file,
        silver_processed_at AS ingestion_timestamp
    FROM seismic_deduplicated
)
-- Visual test and Spark data type confirmation before INSERT
SELECT 
    event_id,
    magnitude,
    typeof(magnitude) AS magnitude_type_info,
    latitude,
    longitude,
    typeof(latitude) AS latitude_type_info,
    depth,
    typeof(depth) AS depth_type_info
FROM typed_data
LIMIT 10;

In [0]:
%sql
WITH typed_data AS (
    SELECT 
        TRIM(event_id) AS event_id,
        magnitude,
        UPPER(TRIM(magnitude_type)) AS magnitude_type,
        magnitude_category AS magnitude_class,
        TRIM(place) AS place,
        event_timestamp,
        longitude,
        latitude,
        depth_km AS depth,
        source_file,
        silver_processed_at AS ingestion_timestamp
    FROM seismic_deduplicated
)
SELECT 
    COUNT(*) AS total_records,
    COUNT(DISTINCT event_id) AS unique_ids,
    COUNT(*) - COUNT(DISTINCT event_id) AS duplicate_count
FROM typed_data;

In [0]:
%sql
SELECT COUNT(*) 
FROM seismic_deduplicated 
WHERE depth_km == 10.0;

In [0]:
%sql
MERGE INTO cr_seismic_analysis.silver.seismic_clean AS target
USING (
    SELECT 
        TRIM(event_id) AS event_id,
        magnitude,
        magnitude_type,
        magnitude_category AS magnitude_class,
        TRIM(place) AS place,
        event_timestamp,
        longitude,
        latitude,
        depth_km AS depth,
        source_file,
        silver_processed_at AS ingestion_timestamp
    FROM seismic_deduplicated
) AS source
ON target.event_id = source.event_id

-- 1. If the earthquake already exists in Silver (e.g., USGS revised the metrics)
WHEN MATCHED THEN
  UPDATE SET
    target.magnitude = source.magnitude,
    target.magnitude_type = source.magnitude_type,
    target.magnitude_class = source.magnitude_class,
    target.place = source.place,
    target.event_timestamp = source.event_timestamp,
    target.longitude = source.longitude,
    target.latitude = source.latitude,
    target.depth = source.depth,
    target.source_file = source.source_file,
    target.ingestion_timestamp = source.ingestion_timestamp

-- 2. If it is a newly recorded earthquake
WHEN NOT MATCHED THEN
  INSERT (
    event_id,
    magnitude,
    magnitude_type,
    magnitude_class,
    place,
    event_timestamp,
    longitude,
    latitude,
    depth,
    source_file,
    ingestion_timestamp
  )
  VALUES (
    source.event_id,
    source.magnitude,
    source.magnitude_type,
    source.magnitude_class,
    source.place,
    source.event_timestamp,
    source.longitude,
    source.latitude,
    source.depth,
    source.source_file,
    source.ingestion_timestamp
  );

In [0]:
%sql
SELECT * FROM cr_seismic_analysis.silver.seismic_clean;

In [0]:
%sql
SELECT 
    (SELECT COUNT(*) FROM seismic_deduplicated) AS temp_view_count,
    (SELECT COUNT(*) FROM cr_seismic_analysis.silver.seismic_clean) AS silver_table_count,
    (SELECT COUNT(*) FROM cr_seismic_analysis.silver.seismic_clean) 
        - (SELECT COUNT(*) FROM seismic_deduplicated) AS difference;

In [0]:
%sql
SELECT 
    seismic_event_id,
    event_id,
    magnitude,
    magnitude_class,
    place,
    depth,
    ingestion_timestamp
FROM cr_seismic_analysis.silver.seismic_clean
ORDER BY seismic_event_id ASC
LIMIT 10;

In [0]:
%sql
SELECT 
    COUNT(*) AS total_records,
    COUNT(CASE WHEN event_id IS NULL THEN 1 END) AS null_event_ids,
    COUNT(CASE WHEN magnitude <= 0 THEN 1 END) AS invalid_magnitudes,
    COUNT(CASE WHEN depth < 0 THEN 1 END) AS invalid_depths,
    COUNT(CASE WHEN latitude NOT BETWEEN 8.0 AND 11.5 
                 OR longitude NOT BETWEEN -86.0 AND -82.5 THEN 1 END) AS out_of_bounds_geo
FROM cr_seismic_analysis.silver.seismic_clean;